# Усиленное восприятие — безопасный пилот
Включите GPU. На Drive/MyDrive/mct положите mct_enhanced_kit.zip и пять camera_new_*.mkv.
Сначала выполните пилот и посмотрите снимки. Полный проход запускается отдельно, после изменения RUN_FULL.
Базовый результат остаётся внутри комплекта и не заменяется.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Выберите GPU в настройках среды'
print(torch.cuda.get_device_name(0))


In [ ]:
import zipfile, shutil, os, subprocess
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/mct')
ROOT = Path('/content/mct_enhanced')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(DRIVE / 'mct_enhanced_kit.zip') as z:
    z.extractall(ROOT)
VIDEOS = Path('/content/mct_videos')
VIDEOS.mkdir(exist_ok=True)
for camera in range(1, 6):
    name = f'camera_new_{camera}.mkv'
    source = DRIVE / name
    assert source.exists(), f'Нет записи: {source}'
    if not (VIDEOS/name).exists() or (VIDEOS/name).stat().st_size != source.stat().st_size:
        shutil.copy2(source, VIDEOS/name)
subprocess.run(['pip','install','-q','boxmot==26.0.0','ultralytics==8.4.203','opencv-python==4.11.0.86','imageio-ffmpeg','lap','pycocotools','pyarrow','pandas','pyyaml','scipy','matplotlib'],check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT/'src')
os.environ['PYTHONUTF8'] = '1'
import yaml
cfg = yaml.safe_load((ROOT/'configs/enhanced.yaml').read_text())
cfg['model'] = str(ROOT/'weights/yolo11m-seg.pt')
(ROOT/'configs/colab_candidate.yaml').write_text(yaml.safe_dump(cfg))
BASE = ROOT/'artifacts/perception'
PILOT = DRIVE/'pilot_v1'
CANDIDATE = DRIVE/'perception_candidate_v1'


## 1. Пилот — выполните сейчас

In [ ]:
import sys
command = [sys.executable,'scripts/run_enhanced.py',str(VIDEOS),'--baseline',str(BASE),'--config','configs/colab_candidate.yaml','--device','0']
subprocess.run(command + ['--out',str(PILOT),'--pilot'],check=True)


In [ ]:
import json
from IPython.display import display, Image
estimate = 0
for summary in sorted(PILOT.glob('*_summary.json')):
    result = json.loads(summary.read_text())
    print(summary.name, result)
    # Conservative initial estimate: pilot includes ROI work even where full run may skip it.
    estimate += result['seconds'] * (20*60/30)
print(f'Ориентир полного прохода: {estimate/60:.0f} минут. Это оценка, не гарантия; сохраните час на проверку.')
for image in sorted(PILOT.glob('*preview*.jpg')):
    print(image.name)
    display(Image(filename=str(image), width=900))
shutil.make_archive(str(DRIVE/'pilot_v1'), 'zip', PILOT)
print('Передайте в чат pilot_v1.zip или снимки и summary. Проверьте дубли, двери, дальний коридор.')


## 2. Полный проход — только после проверки пилота
Если пилот принят и хватает времени, измените RUN_FULL на True и запустите эту ячейку. При обрыве повторите: завершённые камеры проверяются по контрольным суммам.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    subprocess.run(command + ['--out',str(CANDIDATE)],check=True)
    output = shutil.make_archive(str(DRIVE/'perception_candidate_v1'), 'zip', CANDIDATE)
    print('Готово:', output, '\nСкачайте архив и передайте в чат. Исходные artifacts/perception не заменяйте.')
else:
    print('Полный проход не запущен. Сначала проверьте пилот.')
